In [ ]:
from tuutrag.memgraph import MemgraphConnection
import configparser
from pathlib import Path
from tuutrag.data import D
import json 
from tuutrag.types import *



In [ ]:
config = configparser.ConfigParser()
config.read("../config.ini")

port = config.get("Memgraph", "port")
frontend_port = config.get("Memgraph", "frontend_port")
host = config.get("Memgraph", "host")

conn = MemgraphConnection(port=port, frontend_port=frontend_port, host=host)

In [ ]:
data_file: Path = D.processed / "cross_artifact_relations.json"

parts: list[list[str]] = []

with open(data_file, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)

        # Handle object(s)
        if isinstance(data, dict):
            data = [data]

        skipped = 0
        for obj in data:
            for rel in obj.get("relationships", []):
                if isinstance(rel, list):
                    if len(rel) == 3:
                        parts.append([item.strip() for item in rel])
                    else:
                        skipped += 1
                        print(f"Skipping invalid relationship: {rel}")
                elif isinstance(rel, str):
                    part = [item.strip() for item in rel.split(",", 2)]
                    if len(part) == 3:
                        parts.append(part)
                    else:
                        skipped += 1
                        print(f"Skipping invalid relationship: {rel}")
                else:
                    skipped += 1
                    print(f"Skipping invalid relationship: {rel}")

    except Exception as e:
        print(f"Error reading {data_file}: {e}")

    print(f"Extracted {len(parts)} relationships")
    print(f"Skipped {skipped} relationships due to invalid format")

In [ ]:
# Upsert data into Memgraph
# conn.upsert(parts)
# conn.upsert_global(parts)
conn.upsert_universal(parts)

Memgraph Query: "MATCH (n:Entity)-[r:RELATIONSHIP]->(m:Entity)
RETURN n, r, m"